# Code and Data Are Fundamentally the Same

## 1. First Principle: Everything in RAM Is Just Bytes

From the perspective of RAM, **code and data are fundamentally the same thing: sequences of bytes.**

The distinction comes from **how the CPU interprets those bytes**, not from the memory itself.

For example, consider these bytes:

```text
48 83 C0 01
```

As **data**, they are simply four numbers.

As **x86-64 machine code**, they represent:

```asm
add rax, 1
```

If the CPU's instruction pointer (`RIP`) points to those bytes, the CPU attempts to decode and execute them as instructions.

**The CPU does not know whether the programmer intended those bytes to be code or data.**

This is conceptually similar to Lisp's *code-as-data* philosophy. The difference is that Lisp operates on structured program expressions, while the CPU operates on machine-code bytes.

## 2. Can the CPU Execute Data?

**Yes. If memory is executable, the CPU can attempt to execute whatever bytes are stored there.**

Modern CPUs use page-table permissions to control memory access:

| Permission | Meaning |
|---|---|
| Read (R) | Can read bytes |
| Write (W) | Can modify bytes |
| Execute (X) | Can fetch and execute instructions |

A typical memory layout:

| Region | Permissions |
|---|---|
| Code | Read + Execute |
| Data | Read + Write |
| Stack | Read + Write |

What happens if the CPU jumps to a data address?

1. **Executable page:** The CPU attempts to interpret the bytes as instructions.
2. **Non-executable page (NX):** The CPU raises a Page Fault.
3. **Invalid machine instruction:** The CPU may raise an Invalid Opcode Exception (`#UD`).
4. **Valid machine instruction:** The CPU executes it, even if the bytes were originally intended as data.

**The CPU enforces permissions, not programmer intentions.**

## 3. Why This Matters: Security

Historically, attackers could exploit programs by:

1. Writing malicious machine code into a writable memory region, such as the stack.
2. Redirecting execution to that region.
3. Getting the CPU to execute the injected bytes.

Modern systems mitigate this using **NX (No Execute)** and the **W^X principle (Writable XOR Executable)**.

The idea is simple:

> Memory should generally be writable or executable, but not both simultaneously.

This makes code-injection attacks significantly harder.

## 4. The Surprising Consequence: Programs Can Generate Programs

Because machine code is just bytes, a running program can generate new machine code dynamically.

This is the foundation of **Just-In-Time (JIT) compilation**.

```text
Generate machine-code bytes
           ↓
Write bytes into memory
           ↓
Make the memory executable
           ↓
Jump to that address
           ↓
CPU executes the generated code
```

Modern JavaScript engines and JVMs use this fundamental idea.
